In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn, tensor
from torch.utils.data import DataLoader, TensorDataset


class Autoencoder(nn.Module):
    def __init__(self, n_features: int, latent_dim: int = 8, hidden=(32, 16)):
        super().__init__()
        enc = []
        prev = n_features
        for h in hidden:
            enc += [nn.Linear(prev, h), nn.ReLU()]
            prev = h 
        enc.append(nn.Linear(prev, latent_dim))
        self.encoder = nn.Sequential(*enc)

        dec = []
        prev = latent_dim
        for h in reversed(hidden):
            dec +=[nn.Linear(prev, h), nn.ReLU()]
            prev = h 
        dec.append(nn.Linear(prev, n_features))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def train_ae(X_normal: pd.DataFrame, n_features: int, epochs: int = 100, batch_size=512, lr=1e-3, latent_dim: int = 8, hidden: tuple = (32, 16), device: str = 'cpu', seed=131):
    torch.manual_seed(seed)
    model = Autoencoder(n_features=n_features, latent_dim=latent_dim, hidden=hidden).to(device)
    opt = torch.optim.Adam(params=model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    X_arr = X_normal.values if hasattr(X_normal, 'values') else X_normal

    ds1 = TensorDataset(tensor(X_arr, dtype=torch.float32))
    dl = DataLoader(ds1, batch_size=batch_size, shuffle=True)

    for _ in range(epochs):
        for (xb,) in dl:
            xb = xb.to(device)
            opt.zero_grad()
            out=model(xb)
            loss=loss_fn(out, xb)
            loss.backward()
            opt.step()
    model.eval()
    return model


@torch.no_grad()
def get_anomaly_scores(model, X, device = 'cpu', batch_size=4096):
    model.eval()
    X_arr = X.values if hasattr(X, 'values') else X 
    X_tensor = tensor(X_arr, dtype=torch.float32)


    scores = np.zeros(len(X_tensor), dtype=np.float32)

    for start in range(0, len(X_tensor), batch_size):
        xb = X_tensor[start: start + batch_size].to(device)
        out = model(xb)

        err = ((out - xb) ** 2).mean(dim=1).cpu().numpy()
        scores[start: start + batch_size] = err

    return scores


def predict_anomalies(model, X, threshold, device='cpu'):
    scores = get_anomaly_scores(model, X, device)
    predictions = (scores > threshold).astype(int)
    return predictions, scores 


In [ ]:
import optuna
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

from features import feature_engeneering, get_fe_parametrs
from folds import FOLDS, X, y


def objective_ae(trial):
    params = {
        'latent_dim': trial.suggest_categorical('latent_dim', [2, 4, 8, 16, 32]),
        'hidden': trial.suggest_categorical('hidden', [(32, 16), (64, 32), (128, 64), (32,)]),
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [256, 512, 1024]),
        'epochs': 100
    }
    aucs = []
    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)
        
        X_tr_f = feature_engeneering(
            df=X_tr, qt=qt, train_median=train_median,
            train_mad=train_mad, tsh_quantile=tsh_q,
            fti_quantile=fti_q, tt4_quantile=tt4_q
        )
                
        X_val_f = feature_engeneering(
                    df=X_val, qt=qt, train_median=train_median,
                    train_mad=train_mad, tsh_quantile=tsh_q,
                    fti_quantile=fti_q, tt4_quantile=tt4_q
        )

        scaler = StandardScaler()
        X_tr_normal = X_tr_f[y_tr == 0]

        scaler.fit(X_tr_normal)

        X_tr_scaled = scaler.transform(X_tr_normal)
        X_val_scaled = scaler.transform(X_val_f)

        model = train_ae(
            X_normal=X_tr_scaled,
            n_features=X_tr_f.shape[1],

            **params,
            device='cpu'
        )

        val_scores = get_anomaly_scores(model, X_val_scaled, device='cpu')

        pr_auc = average_precision_score(y_val, val_scores)
        aucs.append(pr_auc)

        trial.report(np.mean(aucs), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(aucs))

study_ae = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    sampler=optuna.samplers.TPESampler(seed=131)
)
study_ae.optimize(objective_ae, n_trials=30)

print(f'Best_params: {study_ae.best_params}')
print(f'Best value: {study_ae.best_value}')


c:\Users\DNS\PycharmProjects\ds&ml\anomaly_and_pca\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-21 23:59:03,349] A new study created in memory with name: no-name-7e0a0f48-d05f-45bc-877b-754c20fec745
c:\Users\DNS\PycharmProjects\ds&ml\anomaly_and_pca\venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (32, 16) which is of type tuple.
  optuna_warn(message)
c:\Users\DNS\PycharmProjects\ds&ml\anomaly_and_pca\venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (64, 32) which is of type tuple.
  optuna_warn(message)
c:\Users\

Best_params: {'latent_dim': 16, 'hidden': (32,), 'lr': 0.008326973222958415, 'batch_size': 512}
Best value: 0.5300917666524191


In [3]:

best_params = study_ae.best_params
oof_ae = np.zeros(len(X))
synth_ae = []

for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

    X_tr_f = feature_engeneering(
        df=X_tr, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q, 
        fti_quantile=fti_q, tt4_quantile=tt4_q 
    )
    X_val_f = feature_engeneering(
        df=X_val, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_tr_normal = X_tr_f[y_tr == 0]

    scaler = StandardScaler()
    scaler.fit(X_tr_normal)

    X_tr_scaled = scaler.transform(X_tr_normal)
    X_val_scaled = scaler.transform(X_val_f)

    X_synth = X_tr_normal.copy()
    for col in X_tr_normal.columns:
        X_synth[col] = np.random.permutation(X_synth[col].values)
    X_synth_scaled = scaler.transform(X_synth)

    model = train_ae(
        X_normal=X_tr_normal,
        n_features=X_tr_normal.shape[1],
        **best_params,
        device='cpu'
    )

    oof_ae[val_idx] = get_anomaly_scores(model=model, X=X_val_scaled, device='cpu')

    synth_scores_fold = get_anomaly_scores(model=model, X=X_synth_scaled, device='cpu')
    synth_ae.extend(synth_scores_fold)

print(f'Global PR-AUC: {average_precision_score(y, oof_ae):.4f}')
np.save('oof_ae.npy', oof_ae)
np.save('synth_ae.npy', synth_ae)

Global PR-AUC: 0.3957
